# Fetch one particular Cutout

- author : Sylvie Dagoret-Campagne
- creation date : 2026-06-12

In [ ]:
import io
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import maximum_filter, label
from scipy.optimize import curve_fit
from scipy.interpolate import griddata
from scipy.ndimage import rotate

from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u


from astropy.io import fits
from astropy.visualization import ZScaleInterval
from astropy.visualization.mpl_normalize import ImageNormalize


from astropy.coordinates import AltAz, SkyCoord
from astropy.time import Time
from astropy import units as u
from astropy.coordinates import EarthLocation

In [ ]:
def parallactic_angle(obs_time, ra, dec, location):
    from astropy.coordinates import AltAz, SkyCoord

    target = SkyCoord(ra=ra, dec=dec, unit=u.rad)
    altaz = target.transform_to(AltAz(obstime=obs_time, location=location))

    # hour angle
    lst = obs_time.sidereal_time("apparent", longitude=location.lon)
    ha = (lst - target.ra).to(u.rad)

    lat = location.lat.to(u.rad)
    dec = target.dec.to(u.rad)

    q = np.arctan2(np.sin(ha), np.tan(lat) * np.cos(dec) - np.sin(dec) * np.cos(ha))

    return q

$$\begin{pmatrix} \Delta \alpha \\ \Delta \delta \end{pmatrix} = \mathbf{PC} \cdot \begin{pmatrix} x - CRPIX1 \\ y - CRPIX2 \end{pmatrix}$$

In [ ]:
def plot_cutout_wcs_with_directions(fits_file):

    # --- Load FITS ---
    hdul = fits.open(fits_file)
    data = hdul[0].data
    hdr = hdul[0].header

    wcs = WCS(hdr)

    # --- Time (robuste) ---
    timesys = hdr.get("TIMESYS", "tai")
    if isinstance(timesys, str):
        timesys = timesys.lower()
    else:
        timesys = "tai"

    obstime = Time(hdr["MJD-OBS"], format="mjd", scale=timesys)

    # --- Observatory ---
    loc = EarthLocation(
        lat=hdr.get("OBS-LAT", -30.2446) * u.deg,
        lon=hdr.get("OBS-LONG", -70.7494) * u.deg,
        height=hdr.get("OBS-ELEV", 2663) * u.m,
    )

    # --- Rotation ---
    rotpa = hdr.get("ROTPA", -99999) * u.deg

    # --- Image center ---
    ny, nx = data.shape
    x0, y0 = nx / 2, ny / 2

    ra0, dec0 = wcs.pixel_to_world_values(x0, y0)
    sky_center = SkyCoord(ra=ra0 * u.deg, dec=dec0 * u.deg, frame="icrs")

    print("sky_center = ", ra0, dec0, sky_center)

    # ==========================================================
    # 🔥 CORRECTION MAJEURE : repère tangent local
    # ==========================================================
    from astropy.coordinates import SkyOffsetFrame

    local_frame = SkyOffsetFrame(origin=sky_center)

    print("local_frame", local_frame)

    # --- Directions physiques propres ---
    north = SkyCoord(0 * u.arcsec, +10 * u.arcsec, frame=local_frame).icrs
    east = SkyCoord(+10 * u.arcsec, 0 * u.arcsec, frame=local_frame).icrs

    altaz = sky_center.transform_to(AltAz(obstime=obstime, location=loc))
    alt = altaz.alt
    az = altaz.az
    zenith_distance = 90 * u.deg - altaz.alt
    q_deg = parallactic_angle(obstime, sky_center.ra, sky_center.dec, loc).to_value(u.deg)
    q_rad = parallactic_angle(obstime, sky_center.ra, sky_center.dec, loc).to_value(u.rad)

    print(f"altitude_angle    = {alt}")
    print(f"zenith_angle      = {zenith_distance}")
    print(f"azimuth_angle     = {az}")
    print(f"parallactic_angle = {q_deg}")
    print(f"rotpa_angle       = {rotpa}")

    zenith_altaz = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=obstime, location=loc))

    zenith_local = zenith_altaz.transform_to(local_frame)

    # --- Conversion pixel robuste ---
    def world_to_vec(coord):
        x, y = wcs.world_to_pixel_values(coord.ra.deg, coord.dec.deg)
        return x - x0, y - y0

    vx_n, vy_n = world_to_vec(north)
    vx_e, vy_e = world_to_vec(east)

    vx_z = np.cos(q_rad) * vx_n + np.sin(q_rad) * vx_e
    vy_z = np.cos(q_rad) * vy_n + np.sin(q_rad) * vy_e

    # vx_z, vy_z = world_to_vec(zenith)
    print("north : vx_n, vy_n  = ", vx_n, vy_n)
    print("east  : vx_e, vy_e  = ", vx_e, vy_e)
    print("zenith: vx_z, vy_z  = ", vx_z, vy_z)

    # --- Plot ---
    fig = plt.figure(figsize=(6, 6))
    ax = plt.subplot(projection=wcs)

    ax.imshow(data, origin="lower", cmap="viridis")
    # ax.imshow(data, origin='lower', cmap='gray', transform=ax.get_transform(wcs))

    ax.coords.grid(True, color="white", ls="dotted", lw=2)
    ax.set_xlabel("pixel x (WCS projection)")
    ax.set_ylabel("pixel y (WCS projection)")

    # ==========================================================
    # arrows (attention: échelle arbitraire)
    # ==========================================================
    scale = 0.1
    scalez = scale * np.cos(alt.to_value(u.rad))

    ax.arrow(x0, y0, vx_n * scale, vy_n * scale, color="red", width=0.3, label="North")

    ax.arrow(x0, y0, vx_e * scale, vy_e * scale, color="blue", width=0.3, label="East")

    ax.arrow(x0, y0, vx_z * scale, vy_z * scalez, color="orange", width=0.3, label="Zenith")

    ax.legend()
    plt.title("Cutout with WCS directions")
    plt.show()

### **Get the cutouts**

In [ ]:
objsid = {
    # 0: 313888627167330394,   # rank 1  COSMOS  dipole_frac=0.962  n_dipoles=507 most dipoles before march 2024
    0: 313985344866353157,  # rank 2  COSMOS, positive and negative fluxes, small dipoles, very interesting (many bands)
    1: 313853517840777344,  # rank 3  COSMOS  positive and negative fluxes, small dipoles, very interesting
    2: 313972182542712999,  # rank 4  COSMOS  positive and negative fluxes, high and low dipoles, very interesting
    3: 313871013109563545,  # rank 5 COSMOS; positive and negative fluxes, high and low dipoles, very interesting
    4: 313871013420466334,  # rank 6 COSMOS; positive and negative fluxes, h low dipoles, very interesting
    5: 313998569477505082,  # rank 5 COSMOS:  positive and negative fluxes, h low dipoles
    6: 313994141002367046,  # rank 6 COSMOS: positive and negative fluxes, QSO, 2 types of dipoles in g band few)
    7: 313888627167330394,
    # 0: 170019717267849383, # QSO, no dipole after march 2026
    # 0: 313923786993827880 # before match 2026
}

In [ ]:
DIAOBJECT_IDX = 3  # ← change index to switch object
DIAOBJECT_ID = objsid[DIAOBJECT_IDX]

In [ ]:
diaObjectId = str(DIAOBJECT_ID)

# Get all diaSourceId associated to diaObjectId = 313980948857749506
r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/sources",
    json={
        "diaObjectId": diaObjectId,
        "columns": "r:diaSourceId, r:midpointMjdTai, r:ra, r:raErr, r:dec, r:decErr, r:apFlux, r:apFluxErr, r:scienceFlux, r:scienceFluxErr, r:templateFlux, r:templateFluxErr, r:band",
    },
)

if r.status_code == 200:
    data = pd.read_json(io.BytesIO(r.content))
else:
    print(f"Error for diaObjectId {diaObjectId}: {r.status_code} - {r.text}")

data.to_csv(f"data_{diaObjectId}.csv", index=False)
display(data)

In [ ]:
# Plot scienceFlux light curve
plt.figure(figsize=(10, 5))
plt.errorbar(data["r:midpointMjdTai"], data["r:scienceFlux"], yerr=data["r:scienceFluxErr"], fmt="o")
plt.xlabel("MJD_Tai")
plt.ylabel("scienceFlux (nJy)")
plt.title(f"Light curve - diaObjectId: {diaObjectId}")
plt.grid(True)
plt.show()

In [ ]:
for id in range(len(data["r:diaSourceId"])):
    src = str(data["r:diaSourceId"][id])
    mjd_val = data["r:midpointMjdTai"][id]

    obstime = Time(mjd_val, format="mjd", scale="tai")
    mjd_str = str(mjd_val).replace(".", "_")

    print(f"Processing diaSourceId: {src}, mjd: {mjd_str}")

    for kind in ["Science", "Template", "Difference"]:
        r_cut = requests.post(
            "https://api.lsst.fink-portal.org/api/v1/cutouts",
            json={
                "diaSourceId": src,
                "kind": kind,
                "output-format": "FITS",
            },
        )

        if r_cut.status_code == 200 and len(r_cut.content) > 0:
            try:
                with fits.open(io.BytesIO(r_cut.content), ignore_missing_simple=True) as data_cut:
                    hdr = data_cut[0].header

                    # 🔴 AJOUT TEMPS
                    hdr["MJD-OBS"] = (mjd_val, "Observation midpoint in MJD (TAI)")
                    hdr["TIMESYS"] = ("TAI", "Time system")

                    # Optionnel mais très utile
                    hdr["DATE-OBS"] = (obstime.utc.isot, "UTC ISO time")
                    hdr["COMMENT"] = "Time added from Fink alert (midpointMjdTai)"

                    hdr["OBS-LAT"] = (-30.2446, "Rubin latitude (deg)")
                    hdr["OBS-LONG"] = (-70.7494, "Rubin longitude (deg)")
                    hdr["OBS-ELEV"] = (2663, "Rubin elevation (m)")

                    filename = f"{mjd_str}_cutout_{kind}.fits"
                    data_cut.writeto(filename, overwrite=True)

                    print(f"Saved {filename}")

            except Exception as e:
                print(f"Error processing FITS content for diaSourceId {src} and kind {kind}: {e}")
        else:
            print(f"No FITS content available for diaSourceId {src} and kind {kind}")

    break

In [ ]:
selected_files = f"{mjd_str}_cutout"

In [ ]:
file_diff = f"{selected_files}_Difference.fits"
file_sci = f"{selected_files}_Science.fits"
file_temp = f"{selected_files}_Template.fits"

$$\begin{pmatrix} \Delta \alpha \\ \Delta \delta \end{pmatrix} = \mathbf{PC} \cdot \begin{pmatrix} x - CRPIX1 \\ y - CRPIX2 \end{pmatrix}$$

In [ ]:
# Read the images
with fits.open(file_diff) as hdu:
    header = hdu[0].header
    fits_data = hdu[0].data

display(header)

In [ ]:
header

In [ ]:
loc = EarthLocation.of_site("Rubin Observatory")
time = Time(obstime)

zenith = SkyCoord(alt=90 * u.deg, az=0 * u.deg, frame=AltAz(obstime=time, location=loc))
zenith_icrs = zenith.icrs

In [ ]:
fits_file = file_diff

In [ ]:
# --- Load FITS ---
hdul = fits.open(fits_file)
data = hdul[0].data
hdr = hdul[0].header
wcs = WCS(hdr)

In [ ]:
plt.imshow(data, origin="lower")

$$\begin{pmatrix} \Delta \alpha \\ \Delta \delta \end{pmatrix} = \mathbf{PC} \cdot \begin{pmatrix} x - CRPIX1 \\ y - CRPIX2 \end{pmatrix}$$

In [ ]:
print(wcs)

In [ ]:
plot_cutout_wcs_with_directions(file_diff)